# Explicitly configured dual-I2C glitch experiments
Fill and authorize the fixture configuration before executing hardware cells; never connect GPIO directly to a power rail. Keep the voltage-glitch power stage disabled until digital timing has been measured. Open-drain requires external pull-up, open-source external pull-down; every output needs safe external bias through power loss/BOOTSEL. Opening this notebook does not connect or drive hardware. Protocol 3 is required. Software models do not certify physical timing or electrical safety. See the [glitch notebook guide](../docs/notebook.md) for fixture gates and measurement requirements.

In [ ]:
from rp2040_glitch import (Device, Delay, Pulse, SetPin, Pwm, Write, Poll, WaitI2C, WaitSPI, WaitGPIO, seconds_to_cycles)
PORT = None
BUSES = None  # {0: {"sda": ..., "scl": ...}, 1: {...}}
TARGETS = None  # {0: {"address": ..., "register": bytes(...)}, 1: {...}}
GPIO_CONFIG = None  # {pin: {"drive": "push_pull" / "open_drain" / "open_source", "idle": 0 or 1}}
reset_pin = None  # User fixture label: configure open_drain, idle=1 with external pull-up
glitch_pin = None  # User fixture label: choose drive/idle for your external circuit
SYNC_PIN = None  # Reserved unconnected internal PIO1-to-PIO0 match latch; only hardware waits
SPI = None  # {"cs": ..., "sck": ..., "data": ..., "mode": 0, "lsb_first": False}
AUTHORIZED = False  # Set only after checking wiring, voltage, pull-ups and inactive bias


In [ ]:
assert AUTHORIZED and PORT is not None, "Authorize wiring and select the serial-number-specific CDC port"
device = Device(PORT)
info = device.info()
info

In [ ]:
assert AUTHORIZED and BUSES and TARGETS, "Fill and authorize both bus fixtures first"
for bus in (0, 1):
    device.configure_bus(bus, **BUSES[bus])
    print(bus, device.read_register(bus, TARGETS[bus]["address"], TARGETS[bus]["register"]))

## Register writes and masked active polling
Set explicit data/mask/value for your fixture. Poll NACK means not ready; arbitration loss and bus timeout are terminal. Register pointers are bytes with no implicit byte order.

In [ ]:
WRITE_DATA = None
POLL_MASK = None
POLL_VALUE = None
assert AUTHORIZED and all(x is not None for x in (WRITE_DATA, POLL_MASK, POLL_VALUE))
target = TARGETS[0]
device.write_register(0, target["address"], target["register"], WRITE_DATA)
device.run([Poll(0, target["address"], target["register"], len(POLL_MASK), POLL_MASK, POLL_VALUE)])

## Generic GPIO logic-analyzer pulse example
Choose two verified nonadjacent pins in GPIO_CONFIG and assign the fixture labels reset_pin/glitch_pin; neither is a firmware role. Configure the first as open_drain idle=1, and choose the second's drive/idle for its circuit. GPIO0–29 need not be adjacent; bus/PWM/sync/SPI collisions are rejected. Push-pull drives both levels, open-drain only low or HiZ, open-source only high or HiZ. Pulse asserts opposite configured idle then restores it, not a prior SetPin state; all other pins retain held states. SetPin low/high respect drive mode, while floating means HiZ without pulls. Observe both pins with a logic analyzer and keep the power stage disabled. No sync reservation is needed here.

Counts are system-clock cycles. Required eight-cycle spacing is implicit, including after a wait before a following GPIO transition; explicit delay counts toward the minimum instead of adding another eight. Pulse widths and positive explicit delays below eight remain invalid. Keep intentional delays such as 2500 below. Status inserted_cycles counts padding and programmed_cycles includes it, excluding variable wait time. Optional measured clock calibration affects conversion only; SDK reporting is not scope calibration.

In [ ]:
assert AUTHORIZED and GPIO_CONFIG and reset_pin is not None and glitch_pin is not None
assert abs(reset_pin - glitch_pin) > 1, "Choose nonadjacent fixture pins for this example"
assert GPIO_CONFIG[reset_pin] == {"drive": "open_drain", "idle": 1}
assert glitch_pin in GPIO_CONFIG
for pin, settings in GPIO_CONFIG.items():
    device.configure_gpio(pin=pin, **settings)
MEASURED_CLOCK_HZ = None
clock_hz = MEASURED_CLOCK_HZ or info["sys_hz"]
example = [Pulse(reset_pin, 12500), Delay(2500), Pulse(glitch_pin, 25)]
device.run(example)

## Hardware-timed 500-kHz PWM: 60%, reset 100 ms, delay 10 µs, 10%, delay 100 µs, 60%
Choose a verified unused PWM pin separately. This example needs only the open-drain fixture reset GPIO; PWM-only routines require no generic GPIO or sync configuration. Duty events must be on whole PWM periods (2 µs at exactly 500 kHz). Firmware quantizes frequency to an integer system-cycle period, 16..32768 cycles; actual Hz is sys_hz / period_cycles. Percentages quantize to basis points, ties to even; high cycles round nearest, ties upward. Intermediate duty needs high >=3 and low >=10 cycles; 0%/100% are constant levels. Unsupported grid/duty settings are rejected, not stretched.

Every PWM-containing block restarts carrier phase. Hardware waits cannot share that contiguous timed block, including PWM-pin SetPin; Write/Poll separates blocks. PWM events use the GPIO-adjusted timeline including implicit spacing, and must remain on whole-period boundaries. Success holds final GPIO states and PWM mode/duty by default across bus operations and later runs. hold_final=False restores configured GPIO idle/PWM low. Abort, run errors, disconnect/DTR loss and BOOTSEL always restore idle/PWM low; close drops DTR even after success. clear resets staging/results, not held outputs. External bias covers power loss/ROM release. Software models do not certify physical timing: physical use requires measurements of frequency, duty/mode boundaries, floating behavior, reset edges and shutdown on the authorized fixture.

In [ ]:
PWM_PIN = None  # Explicit unused exposed GPIO, distinct from all configured pins
PWM_AUTHORIZED = False
assert AUTHORIZED and PWM_AUTHORIZED and reset_pin is not None and PWM_PIN is not None
device.configure_gpio(pin=reset_pin, drive="open_drain", idle=1)
device.configure_pwm(pin=PWM_PIN, hz=500_000)
pwm_clock_hz = info["sys_hz"]  # May substitute a measured clock; cycle-grid checks still apply
device.run([
    Pwm(60),
    Pulse(reset_pin, seconds_to_cycles(0.100, clock_hz=pwm_clock_hz)),
    Delay(seconds_to_cycles(10e-6, clock_hz=pwm_clock_hz)),
    Pwm(10),
    Delay(seconds_to_cycles(100e-6, clock_hz=pwm_clock_hz)),
    Pwm(60),
])
# Final 60% remains running for authorized measurement; run the stop cell next.

In [ ]:
assert AUTHORIZED and PWM_AUTHORIZED and PWM_PIN is not None
device.abort()  # Explicitly stop the held carrier before unrelated examples

## Guarded PWM disable, static states and resume
Authorize all three static states for the same explicitly selected PWM pin before running this cell. No generic GPIO configuration is needed on that pin. SetPin changes PWM mode at hardware period boundaries, not via CPU dispatch; a later Pwm resumes the carrier. The delays below intentionally hold each mode for 50 periods. Final floating stops the PWM stream engine while retaining HiZ without pulls. status/info pwm.final_mode describes the target endpoint (carrier/low/high/floating), not instantaneous live mode; active becomes false after final static completion.

In [ ]:
PWM_STATES_AUTHORIZED = False
assert AUTHORIZED and PWM_AUTHORIZED and PWM_STATES_AUTHORIZED and PWM_PIN is not None
pwm_info = device.info()["pwm"]
assert pwm_info is not None and pwm_info["pin"] == PWM_PIN
mode_hold = 50 * pwm_info["period_cycles"]
pin_state_result = device.run([
    Pwm(60), Delay(mode_hold),
    SetPin(PWM_PIN, "floating"), Delay(mode_hold),
    SetPin(PWM_PIN, "low"), Delay(mode_hold),
    SetPin(PWM_PIN, "high"), Delay(mode_hold),
    Pwm(60), Delay(mode_hold),
    SetPin(PWM_PIN, "floating"),
])
pin_state_result

## Restore idle instead of holding final state
This separate opt-in run demonstrates hold_final=False; after success all generic GPIOs return to configured idle and PWM is low. device.set_pin(pin, state, hold_final=True) is the one-step convenience API and requires prior GPIO or PWM configuration. abort() always restores idle/PWM low, including after a successful held state.

In [ ]:
RESTORE_IDLE_AUTHORIZED = False
assert AUTHORIZED and PWM_AUTHORIZED and PWM_STATES_AUTHORIZED and RESTORE_IDLE_AUTHORIZED
assert PWM_PIN is not None
device.run([Pwm(60), Delay(mode_hold), SetPin(PWM_PIN, "floating")], hold_final=False)
device.set_pin(PWM_PIN, "low", hold_final=False)

## Hardware I2C prefix trigger
Configure SYNC_PIN explicitly as an unused, unconnected pin: PIO1 raises the internal match latch and PIO0 reads it without CPU dispatch; no jumper is needed. Pure GPIO pulses/PWM do not need it. The latch is live at block launch, including during a preceding fixture-reset pulse. A match during that pulse is consumed after its release. To ignore traffic during reset, complete its pulse in a separate run before arming. The final selected bit, not STOP/final ACK, is the trigger reference. Interrupt a waiting cell to abort.

In [ ]:
I2C_PREFIX = None  # bytes, zero to two initial data bytes
assert AUTHORIZED and I2C_PREFIX is not None and SYNC_PIN is not None and GPIO_CONFIG
device.configure_sync(pin=SYNC_PIN)
target = TARGETS[0]
device.run([Pulse(reset_pin, 12500), WaitI2C(0, target["address"], False, I2C_PREFIX), Delay(2500), Pulse(glitch_pin, 25)])

## Selectable SPI or GPIO trigger (uses the same single hardware trigger slot)
Choose SPI prefix matching or a GPIO edge. GPIO is input-only, without internal pulls; select a free, voltage-compatible pin. Rising=True waits low then high; False waits high then low. An already-active level does not fire. Hold each GPIO level at least four system cycles; this is not a debounced switch input. The latch is live from block launch, including during a preceding reset.

For SPI use MOSI or MISO as data. At 10 MHz require SCK high/low >=50 ns, CS setup/hold >=8 system cycles, and inactive gap >=64 cycles. Physical timing still requires electrical validation.

In [ ]:
TRIGGER_SOURCE = None  # "spi" or "gpio"
SPI_PREFIX = None  # bytes, one to four bytes in normal byte representation
GPIO_PIN = None  # GPIO0–29, distinct from every configured bus/output/SPI pin
GPIO_RISING = True
assert AUTHORIZED and GPIO_CONFIG and SYNC_PIN is not None
device.configure_sync(pin=SYNC_PIN)
if TRIGGER_SOURCE == "spi":
    assert SPI and SPI_PREFIX is not None
    device.configure_spi(**SPI)
    trigger = WaitSPI(SPI_PREFIX)
elif TRIGGER_SOURCE == "gpio":
    assert GPIO_PIN is not None
    trigger = WaitGPIO(GPIO_PIN, rising=GPIO_RISING)
else:
    raise ValueError("Choose an explicitly configured trigger source")
device.run([trigger, Pulse(glitch_pin, 25)])

## Explicit parameter sweep
Every call waits for the on-device routine; no USB-timed edges. Enable only after a digital-only fixture check. No automatic mutation retry.

In [ ]:
SWEEP_AUTHORIZED = False
assert AUTHORIZED and SWEEP_AUTHORIZED
results = []
for width in (8, 9, 16, 25):
    result = device.run([Pulse(reset_pin, 12500), Delay(2500), Pulse(glitch_pin, width)])
    results.append((width, result))
results

In [ ]:
device.abort()  # idempotent safety stop
device.close()

## Final opt-in ROM USB programming mode
This does not flash firmware or reset the target. The accepted response closes CDC; verify BOOTSEL with picotool or USB VID:PID 2e8a:0003. An uncertain acknowledgment is not proof of enumeration.

In [ ]:
BOOTSEL_AUTHORIZED = False
assert BOOTSEL_AUTHORIZED and PORT
with Device(PORT) as device:
    device.bootloader()